# Loan Dataset Analysis and Classification

1. Exploratory Data Analysis (EDA)
2. Data Preprocessing
3. Model Training and Evaluation using three classifiers:
   - Decision Tree
   - K-Nearest Neighbors (KNN)
   - Naive Bayes
4. Comparison of Results
5. Kaggle Submission File Generation

---


## Data Loading and Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import ssl
import certifi
import itertools
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.tree import plot_tree

### Load data

Loaded loan dataset and take a quick look at its size and first few rows to get an overview of the data.

In [ ]:
file_path = '../datasets/184-702-tu-ml-2025-w-loan/'
df= pd.read_csv(file_path + "loan-10k.lrn.csv")
df_kaggle_test= pd.read_csv(file_path + "loan-10k.tes.csv")
df= df.set_index("ID")
df_kaggle_test=df_kaggle_test.set_index("ID")
df.head()

In [ ]:
print(df.shape)
print(df_kaggle_test.shape)

### Basic information

Checking basic info about the dataset — column types, summary stats, and any missing values.

In [ ]:
df.info()
df.describe()
df.isna().sum()

In [ ]:
df_kaggle_test.info()
df_kaggle_test.describe()
df_kaggle_test.isna().sum()

### Target Variable Analysis

Plotting the target distribution and feature histograms to see class balance and variable spread.

In [ ]:
target_col = 'grade' 
sns.countplot(x=target_col, data=df)
plt.title('Loan Grade - Training set')
plt.xlabel('Loan Grade')
plt.show()

In [ ]:
df["grade"].describe()

### Feature Exploration

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].hist(figsize=(24, 16))
plt.suptitle("Numeric Feature Distributions")
plt.show()

cat_cols = df.select_dtypes(exclude=np.number).columns

fig, axes = plt.subplots(len(cat_cols), 1, figsize=(12, 4 * len(cat_cols)))

for ax, col in zip(axes, cat_cols):
    sns.countplot(x=df[col], ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.tick_params(axis='x', rotation=45)


plt.tight_layout()
plt.show()

# Preprocessing

## Target varaiable 

The target variable "grade" is given as grades A to G, where A is the best and G the worst grade. Most models require numerical input, so we will map the grades to numerical values as follows: A=0, B=1, C=2, D=3, E=4, F=5, G=6. We use the label encoding here since there is an inherent order in the grades.

In [ ]:

# Create label encoder
le = LabelEncoder()

# Fit on the ordered list of grades to ensure correct mapping
le.fit(['A', 'B', 'C', 'D', 'E', 'F', 'G'])
df['grade'] = le.fit_transform(df['grade'])

sns.countplot(x=target_col, data=df)
plt.title('Loan Grade - Training set')
plt.xlabel('Loan Grade')
plt.show()


## Input variables

### Check for outliers

In [ ]:
# Select numeric columns (excluding target variable)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'grade' in numeric_cols:
    numeric_cols.remove('grade') # class not included

print(f"Number of numeric columns to check: {len(numeric_cols)}")

# Define and run outlier detection function
def detect_outliers_iqr(df, columns):
    
    outlier_info = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
        outlier_percentage = (outlier_count / len(df)) * 100
        
        outlier_info[col] = {
            'count': outlier_count,
            'percentage': outlier_percentage,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'min': df[col].min(),
            'max': df[col].max(),
            'mean': df[col].mean(),
            'std': df[col].std()
        }
    
    return pd.DataFrame(outlier_info).T.sort_values('percentage', ascending=False)

# Run outlier detection
outlier_summary = detect_outliers_iqr(df, numeric_cols)
print("\nOutlier Summary:")
print(outlier_summary.head(30))

# Create the boxplots
n_cols = 5
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    df.boxplot(column=col, ax=axes[idx])
    outlier_pct = outlier_summary.loc[col, 'percentage']
    axes[idx].set_title(f'{col}', fontsize=9)
    axes[idx].tick_params(labelsize=8)


plt.tight_layout()
plt.show()

#### Handle features with meaningful but wide outliers

Log transforming heavily skewed predictors

In [ ]:
# Features to log-transform
skewed_features = ['annual_inc', 'revol_bal', 'total_rec_late_fee', 'recoveries', 
                   'collection_recovery_fee', 'tot_coll_amt', 'total_rev_hi_lim', 
                   'delinq_amnt', 'mort_acc', 'tot_hi_cred_lim', 'total_bal_ex_mort', 
                   'total_bc_limit']

# Apply log1p transformation (log(1+x) to handle zeros)
for col in skewed_features:
    df[f'{col}_log'] = np.log1p(df[col])
    df_kaggle_test[f'{col}_log'] = np.log1p(df_kaggle_test[col])

# Drop original skewed columns
df = df.drop(columns=skewed_features)
df_kaggle_test = df_kaggle_test.drop(columns=skewed_features)


In [ ]:
skewed_features = [f + "_log" for f in skewed_features]

# Create the boxplots
n_cols = 3
n_rows = (len(skewed_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, n_rows * 3))
axes = axes.ravel()

for idx, col in enumerate(skewed_features):
    df.boxplot(column=col, ax=axes[idx])
    axes[idx].set_title(f'{col}', fontsize=9)
    axes[idx].tick_params(labelsize=8)


plt.tight_layout()
plt.show()

#### Handle figor values 0

In [ ]:
df[df['last_fico_range_low']==0][['last_fico_range_high','last_fico_range_low']]

In [ ]:
# Check the typical range width in your data
df_train=df.copy()
df_train['fico_range_width'] = df_train['last_fico_range_high'] - df_train['last_fico_range_low']

# Only calculate for valid rows (both > 0)
valid_ranges = df_train[(df_train['last_fico_range_high'] > 0) & 
                        (df_train['last_fico_range_low'] > 0)]

print("FICO Range Width Statistics:")
print(valid_ranges['fico_range_width'].describe())
print(f"\nMost common width: {valid_ranges['fico_range_width'].mode().values[0]}")

# Check distribution
print("\nWidth distribution:")
print(valid_ranges['fico_range_width'].value_counts().head(10))

we can see that the typical width is 4 for figo scores

In [ ]:
# Handle figor values 0: use the 4 range to fill it up

df= df[(df["last_fico_range_high"]!=0) | (df["last_fico_range_low"]!=0)] # remove all (2) rows wheer both are =0 / we see only low=0 are still there

mask= df['last_fico_range_low']==0
df.loc[mask, 'last_fico_range_low']= df.loc[mask, 'last_fico_range_high']-4 # change all last_fico_range_low=0 valuesd with last_fico_range_high-4

df[df['last_fico_range_low']==0][['last_fico_range_high','last_fico_range_low']].shape



df_kaggle_test= df_kaggle_test[(df_kaggle_test["last_fico_range_high"]!=0) | (df_kaggle_test["last_fico_range_low"]!=0)] # remove all (2) rows wheer both are =0 / we see only low=0 are still there

mask= df_kaggle_test['last_fico_range_low']==0
df_kaggle_test.loc[mask, 'last_fico_range_low']= df_kaggle_test.loc[mask, 'last_fico_range_high']-4 # change all last_fico_range_low=0 valuesd with last_fico_range_high-4

df_kaggle_test[df_kaggle_test['last_fico_range_low']==0][['last_fico_range_high','last_fico_range_low']].shape



#### Handle attributes with no variation

In [ ]:
df["policy_code"].value_counts()
df_kaggle_test["policy_code"].value_counts()

We can see that the attribute "policy_code" has no variation in the training set as well as in the test set. Therefore, we will drop this attribute from both datasets.

In [ ]:
df= df.drop(columns=["policy_code"])
df_kaggle_test= df_kaggle_test.drop(columns=["policy_code"])

#### Handle data (year and month attributes)

In [ ]:
date_data= df[[col for col in df.columns if "year" in col or "month" in col]]
date_data.columns

In [ ]:
# --- Combine month and year columns into datetime columns in both datasets ---

for data in [df, df_kaggle_test]:
    # Convert to numeric safely (handles strings or NaNs)
    for col in ['issue_d_month', 'earliest_cr_line_month', 'last_pymnt_d_month', 'last_credit_pull_d_month']:
        data[col] = pd.to_numeric(data[col])
        data[col] = data[col] + 1  # shift from [0,11] → [1,12]
        data.loc[(data[col] < 1) | (data[col] > 12), col] = np.nan  # ensure valid months

    for col in ['issue_d_year', 'earliest_cr_line_year', 'last_pymnt_d_year', 'last_credit_pull_d_year']:
        data[col] = pd.to_numeric(data[col])

    # Combine into datetime (invalid will become NaT)
    data['issue_d'] = pd.to_datetime(
        data['issue_d_year'].astype('Int64').astype(str) + '-' + data['issue_d_month'].astype('Int64').astype(str) + '-01'
    )
    data['earliest_cr_line'] = pd.to_datetime(
        data['earliest_cr_line_year'].astype('Int64').astype(str) + '-' + data['earliest_cr_line_month'].astype('Int64').astype(str) + '-01'
    )
    data['last_pymnt_d'] = pd.to_datetime(
        data['last_pymnt_d_year'].astype('Int64').astype(str) + '-' + data['last_pymnt_d_month'].astype('Int64').astype(str) + '-01'
    )
    data['last_credit_pull_d'] = pd.to_datetime(
        data['last_credit_pull_d_year'].astype('Int64').astype(str) + '-' + data['last_credit_pull_d_month'].astype('Int64').astype(str) + '-01'
    )
# --- Define a fixed reference date (max issue year from both datasets) ---
reference_year = max(df['issue_d_year'].max(), df_kaggle_test['issue_d_year'].max())
reference_date = pd.Timestamp(year=int(reference_year), month=12, day=1)
print(reference_date)

# --- Create numeric duration features ---
for data in [df, df_kaggle_test]:
    # Duration between earliest credit line and loan issue
    data['credit_history_age_months'] = (
        12 * (data['issue_d'].dt.year - data['earliest_cr_line'].dt.year) +
        (data['issue_d'].dt.month - data['earliest_cr_line'].dt.month)
    )

    # Months since last payment (relative to reference date)
    data['months_since_last_payment'] = (
        12 * (reference_date.year - data['last_pymnt_d'].dt.year) +
        (reference_date.month - data['last_pymnt_d'].dt.month)
    )

    # Months since last credit pull
    data['months_since_credit_pull'] = (
        12 * (reference_date.year - data['last_credit_pull_d'].dt.year) +
        (reference_date.month - data['last_credit_pull_d'].dt.month)
    )

    # Drop original month/year columns (redundant)
    data.drop(columns=[
        'issue_d_month', 'issue_d_year',
        'earliest_cr_line_month', 'earliest_cr_line_year',
        'last_pymnt_d_month', 'last_pymnt_d_year',
        'last_credit_pull_d_month', 'last_credit_pull_d_year'
    ], inplace=True)


The dataset contained several date-related attributes stored as separate month and year columns, encoded with months in the range [0, 11].
To make these features useful for machine learning models, we performed the following preprocessing and feature engineering steps:

Month Conversion and Date Construction

All month columns were shifted by +1 to convert them from [0, 11] to [1, 12].

Each pair of month/year columns was combined into a single datetime column using pandas.to_datetime.

Fixed Reference Date

A fixed reference date (December of the latest issue year in the dataset) was used.

Derived Duration Features

credit_history_age_months: Number of months between the borrower’s earliest credit line and the loan issue date.

months_since_last_payment: Number of months between the last recorded payment and the reference date.

months_since_credit_pull: Number of months since the last credit pull date.

In [ ]:
# Define the new date-derived columns
date_features = [
    'credit_history_age_months',
    'months_since_last_payment',
    'months_since_credit_pull'
]

# Create the boxplots
n_cols = 1
n_rows = (len(date_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5, n_rows * 3))
axes = axes.ravel()

for idx, col in enumerate(date_features):
    df.boxplot(column=col, ax=axes[idx])
    axes[idx].set_title(f'{col}', fontsize=9)
    axes[idx].tick_params(labelsize=8)


plt.tight_layout()
plt.show()

### Handling missing values

In [ ]:
print(df.isnull().sum().value_counts())
print((df.astype(str)=="?").any().value_counts())
print((df.astype(str)=="nan").any().value_counts())

we can see there are no missing values in the dataset.

In [ ]:
print(df_kaggle_test.isnull().sum().value_counts())
print((df_kaggle_test.astype(str)=="?").any().value_counts())
print((df_kaggle_test.astype(str)=="nan").any().value_counts())

### Encoding categorical variables

In [ ]:
#Identify the categorical variables:
cat_cols = df.select_dtypes(exclude=[np.number,np.datetime64]).columns
numeric_cols = df.select_dtypes(np.number).columns
cat_cols

In [ ]:
for col in cat_cols:
    print(col)
    print(df[col].unique())

In [ ]:
#Use specialized lable encoding for emp_length by using the years directly
# Employment length
def emp_to_int(x):
    if x == '< 1 year':
        return 0
    elif x == '10+ years':
        return 10
    else:
        return int(x.split()[0])

df['emp_length'] = df['emp_length'].apply(emp_to_int)
df_kaggle_test['emp_length'] = df_kaggle_test['emp_length'].apply(emp_to_int)


We encode categorical features as numbers so models can process them.

In [ ]:
# Identify binary codable categorical variables
bin_cat =[]
for col in cat_cols:
    if df[col].nunique() == 2:
        bin_cat.append(col)

bin_cat

# ENcode them with lable encoding
df[bin_cat]= df[bin_cat].apply(LabelEncoder().fit_transform)
df[bin_cat].head()

df_kaggle_test[bin_cat]= df_kaggle_test[bin_cat].apply(LabelEncoder().fit_transform)
df_kaggle_test[bin_cat].head()

In [ ]:
# Use 1 hot encoding directly for columnes with more then 2 but not too much unique values
nominal_cols = ['home_ownership', 'verification_status',  'purpose', 'loan_status']

df = pd.get_dummies(df, columns=nominal_cols)
df_kaggle_test = pd.get_dummies(df_kaggle_test, columns=nominal_cols)

In [ ]:
# for states we first group them into broader groups and then use one hot encoding
northeast = ['ME','NH','VT','MA','RI','CT','NY','NJ','PA']
midwest = ['OH','MI','IN','IL','WI','MN','IA','MO','ND','SD','NE','KS']
south = ['DE','MD','DC','VA','WV','NC','SC','GA','FL','KY','TN','MS','AL','AR','LA','OK','TX']
west = ['ID','MT','WY','NV','UT','CO','AZ','NM','AK','WA','OR','CA','HI']

def map_region(state):
    if state in northeast: return 'Northeast'
    elif state in midwest: return 'Midwest'
    elif state in south: return 'South'
    elif state in west: return 'West'
    else: return 'Other'

df['addr_state'] = df['addr_state'].apply(map_region)
df_kaggle_test['addr_state'] = df_kaggle_test['addr_state'].apply(map_region)

df = pd.get_dummies(df, columns=['addr_state'])
df_kaggle_test = pd.get_dummies(df_kaggle_test, columns=['addr_state'])

In [ ]:
for col in cat_cols:
    dummy_cols = [c for c in df.columns if c.startswith(col + '_')]
    print(f"{col} -> {dummy_cols}")

In [ ]:
df.head()

## Split training data into train and test sets

We split the dataset into training and testing sets to evaluate model performance.

In [ ]:
X = df.drop("grade", axis=1)  # features
y = df["grade"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% of data for testing
    random_state=42,     # ensures reproducibility
    stratify=y           # keeps same class proportions in train/test
)

X_train.head()

## Scale data

In [ ]:
numeric_cols= numeric_cols.drop("grade")
numeric_cols = numeric_cols.union(["emp_length"])
#Scale training and test data
X_train_scaled=X_train.copy()
X_test_scaled= X_test.copy()
scaler = StandardScaler()
X_train_scaled[numeric_cols]= scaler.fit_transform(X_train_scaled[numeric_cols])
X_test_scaled[numeric_cols]= scaler.fit_transform(X_test_scaled[numeric_cols])
X_test_scaled.describe()

In [ ]:
# Scale kaggle training set
df_kaggle_test_scaled=df_kaggle_test.copy()
scaler = StandardScaler()
df_kaggle_test_scaled[numeric_cols]= scaler.fit_transform(df_kaggle_test_scaled[numeric_cols])


Exclude time date variables as they can not be hanled by th classifiers

In [ ]:
X_train = X_train.select_dtypes(exclude=["datetime64[ns]"])
X_test  = X_test.select_dtypes(exclude=["datetime64[ns]"])

X_train_scaled = X_train_scaled.select_dtypes(exclude=["datetime64[ns]"])
X_test_scaled  = X_test_scaled.select_dtypes(exclude=["datetime64[ns]"])

df_kaggle_test = df_kaggle_test.select_dtypes(exclude=["datetime64[ns]"])
df_kaggle_test_scaled  = df_kaggle_test_scaled.select_dtypes(exclude=["datetime64[ns]"])

# Model Training and Evaluation

We train three classifiers — Decision Tree, KNN, and Naive Bayes — and check how well they predict loan grades.

## Decision Tree Classifier

In [ ]:

# Hyperparameter lists
max_depths = [5,10, 20,30, None]
min_splits = [5,10,20,30]
min_leafs = [5,10,20,30]
criterions = ["gini", "entropy"]
weights = [None, "balanced"]

results = []

# Loop through all combinations
for depth, split, leaf, crit, weight in itertools.product(max_depths, min_splits, min_leafs, criterions, weights):
    
    clf = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_split=split,
        min_samples_leaf=leaf,
        criterion=crit,
        class_weight=weight,
        random_state=42
    )
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)  # probabilities for all classes
    
    results.append({
        "max_depth": depth,
        "min_samples_split": split,
        "min_samples_leaf": leaf,
        "criterion": crit,
        "class_weight": str(weight),
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average='macro'),
        "weighted_f1": f1_score(y_test, y_pred, average='weighted'),
        "auc": roc_auc_score(y_test, y_prob, multi_class="ovr") # one over rest
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)
res_df_ordered= results_df.copy()
res_df_ordered.sort_values(by="macro_f1", ascending=False, inplace=True)
res_df_ordered = res_df_ordered.set_index(["class_weight","criterion","max_depth","min_samples_split","min_samples_leaf"])
res_df_ordered= res_df_ordered.sort_index()
res_df_ordered.head(50)

In [ ]:
# Plots for visualization

## Headmap for the differnt min/max values:
leaf_values = sorted(results_df["min_samples_leaf"].unique())

# Create a single figure with 1 row and N columns (one subplot per leaf value)
fig, axes = plt.subplots(1, len(leaf_values), figsize=(6 * len(leaf_values), 4), sharey=True)

for ax, leaf in zip(axes, leaf_values):
    subset = results_df[results_df["min_samples_leaf"] == leaf]
    heatmap_data = subset.pivot_table(
        values="macro_f1",
        index="max_depth",
        columns="min_samples_split"
    )
    
    sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"min_samples_leaf = {leaf}")
    ax.set_xlabel("min_samples_split")
    ax.set_ylabel("max_depth")

plt.suptitle("Macro F1 by max_depth and min_samples_split for different min_samples_leaf values", y=1.05)
plt.tight_layout()
plt.show()


## Bar chart to show the effect of differnt criterions
sns.barplot(x='class_weight', y='macro_f1', hue='criterion', data=results_df)
plt.title("Effect of class_weight and criterion on Macro F1")
plt.show()

## Plot to show the influnce of weighting the classes
sns.scatterplot(x='weighted_f1', y='macro_f1', hue='class_weight', style='criterion', data=results_df)
plt.title("Weighted F1 vs Macro F1 across Decision Tree variants (class_weight)")
plt.show()



We numerically search the best model dependent on the various evaluation metrics (macro_f1, weighted_f1, accuracy, auc).:

In [ ]:
# Choose the best combination of hyper paramters model:
hyperparams = ["max_depth","min_samples_split","min_samples_leaf","criterion","class_weight"]
# Pick the row with the highest Macro F1
best_model = results_df.loc[results_df['macro_f1'].idxmax()]

print("Optimal hyperparameter combination based on Macro F1:")
print(best_model[hyperparams])

# Pick the row with the highest weighted_f1
best_model = results_df.loc[results_df['weighted_f1'].idxmax()]

print("Optimal hyperparameter combination based on weighted_f1:")
print(best_model[hyperparams])

# Pick the row with the highest accuracy
best_model = results_df.loc[results_df['accuracy'].idxmax()]

print("Optimal hyperparameter combination based on accuracy:")
print(best_model[hyperparams])

# Pick the row with the highest auc
best_model = results_df.loc[results_df['auc'].idxmax()]

print("Optimal hyperparameter combination based on auc:")
print(best_model[hyperparams])


We fit the choosen best model and look what the most importnat features are:

In [ ]:
# Fit the evaluated optical model and get the most important attributes
clf_best = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=5,
    criterion="gini",
    class_weight=None,
    random_state=42
)

clf_best.fit(X_train, y_train)

# Get feature importances
importances = clf_best.feature_importances_

# Combine with feature names
feature_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print("Top important features:")
print(feature_importance_df.head(10)) 

We compare the evaluation metrics for the unscaled and scaled dataset:

In [ ]:
#Resulting evaulation values for the best selected model
test_pred = clf_best.predict(X_test)

print("Classification report for the UNSCALED data: ")
print(classification_report(y_test, test_pred))


# Apply the best model on the scaled data (shoulnt make any big difference)
clf_best_scaled = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=5,
    criterion="gini",
    class_weight=None,
    random_state=42
)
clf_best_scaled.fit(X_train_scaled, y_train)

test_pred_scaled = clf_best_scaled.predict(X_test_scaled)

print("Classification report for the SCALED data: ")
print(classification_report(y_test, test_pred_scaled))

In [ ]:
#here the tree it too big for a meaningfull representation
plt.figure(figsize=(20, 10))
plot_tree(
    clf_best,
    filled=True,
    rounded=True,
    class_names=[str(cls) for cls in clf_best.classes_],
    feature_names=X_train.columns,
    fontsize=10
)
plt.title("Decision Tree (Best Model)")
plt.show()

## Model Comparison

## Kaggle Submission